In [ ]:
from flask import Flask, render_template, request, redirect, jsonify
import sqlite3
import string
import random
from urllib.parse import urlparse

app = Flask(__name__)

DATABASE = "urls.db"


# ---------------------------------------------------------
# DATABASE FUNCTIONS
# ---------------------------------------------------------

def get_db_connection():
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    return conn


def initialize_database():
    conn = get_db_connection()

    conn.execute("""
        CREATE TABLE IF NOT EXISTS urls (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            original_url TEXT NOT NULL,
            short_code TEXT UNIQUE NOT NULL,
            clicks INTEGER DEFAULT 0,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.commit()
    conn.close()


# ---------------------------------------------------------
# URL VALIDATION
# ---------------------------------------------------------

def is_valid_url(url):
    try:
        parsed = urlparse(url)
        return parsed.scheme in ["http", "https"] and bool(parsed.netloc)
    except Exception:
        return False


# ---------------------------------------------------------
# SHORT CODE GENERATION
# ---------------------------------------------------------

def generate_short_code(length=6):
    characters = string.ascii_letters + string.digits

    while True:
        code = ''.join(random.choices(characters, k=length))

        conn = get_db_connection()
        existing = conn.execute(
            "SELECT id FROM urls WHERE short_code = ?",
            (code,)
        ).fetchone()
        conn.close()

        if existing is None:
            return code


# ---------------------------------------------------------
# HOME PAGE
# ---------------------------------------------------------

@app.route("/")
def home():
    conn = get_db_connection()

    urls = conn.execute("""
        SELECT *
        FROM urls
        ORDER BY created_at DESC
    """).fetchall()

    conn.close()

    return render_template("index.html", urls=urls)


# ---------------------------------------------------------
# CREATE SHORT URL
# ---------------------------------------------------------

@app.route("/shorten", methods=["POST"])
def shorten_url():

    original_url = request.form.get("url", "").strip()

    if not original_url:
        return redirect("/")

    if not is_valid_url(original_url):
        return redirect("/")

    short_code = generate_short_code()

    conn = get_db_connection()

    conn.execute("""
        INSERT INTO urls (original_url, short_code)
        VALUES (?, ?)
    """, (original_url, short_code))

    conn.commit()
    conn.close()

    return redirect("/")


# ---------------------------------------------------------
# REDIRECT SHORT URL
# ---------------------------------------------------------

@app.route("/<short_code>")
def redirect_to_original(short_code):

    conn = get_db_connection()

    url_data = conn.execute("""
        SELECT *
        FROM urls
        WHERE short_code = ?
    """, (short_code,)).fetchone()

    if url_data is None:
        conn.close()
        return "Short URL not found", 404

    conn.execute("""
        UPDATE urls
        SET clicks = clicks + 1
        WHERE short_code = ?
    """, (short_code,))

    conn.commit()
    conn.close()

    return redirect(url_data["original_url"])


# ---------------------------------------------------------
# DELETE SHORT URL
# ---------------------------------------------------------

@app.route("/delete/<short_code>", methods=["POST"])
def delete_url(short_code):

    conn = get_db_connection()

    conn.execute("""
        DELETE FROM urls
        WHERE short_code = ?
    """, (short_code,))

    conn.commit()
    conn.close()

    return redirect("/")


# ---------------------------------------------------------
# REST API
# ---------------------------------------------------------

@app.route("/api/urls", methods=["GET"])
def get_urls():

    conn = get_db_connection()

    urls = conn.execute("""
        SELECT id, original_url, short_code, clicks, created_at
        FROM urls
        ORDER BY created_at DESC
    """).fetchall()

    conn.close()

    result = []

    for url in urls:
        result.append({
            "id": url["id"],
            "original_url": url["original_url"],
            "short_code": url["short_code"],
            "short_url": request.host_url + url["short_code"],
            "clicks": url["clicks"],
            "created_at": url["created_at"]
        })

    return jsonify(result)


# ---------------------------------------------------------
# API: CREATE SHORT URL
# ---------------------------------------------------------

@app.route("/api/shorten", methods=["POST"])
def api_shorten():

    data = request.get_json()

    if not data or "url" not in data:
        return jsonify({
            "error": "URL is required"
        }), 400

    original_url = data["url"].strip()

    if not is_valid_url(original_url):
        return jsonify({
            "error": "Invalid URL"
        }), 400

    short_code = generate_short_code()

    conn = get_db_connection()

    conn.execute("""
        INSERT INTO urls (original_url, short_code)
        VALUES (?, ?)
    """, (original_url, short_code))

    conn.commit()
    conn.close()

    return jsonify({
        "original_url": original_url,
        "short_code": short_code,
        "short_url": request.host_url + short_code
    }), 201


# ---------------------------------------------------------
# APPLICATION START
# ---------------------------------------------------------

if __name__ == "__main__":

    initialize_database()

    app.run(
        host="127.0.0.1",
        port=5000,
        debug=True
    )

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


In [ ]:
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="UTF-8">

    <meta name="viewport"
          content="width=device-width, initial-scale=1.0">

    <title>URL Shortener</title>

    <style>

        body {
            font-family: Arial, sans-serif;
            background: #f4f6f8;
            margin: 0;
            padding: 40px;
        }

        .container {
            max-width: 900px;
            margin: auto;
            background: white;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0 3px 12px rgba(0,0,0,0.1);
        }

        h1 {
            text-align: center;
            margin-bottom: 30px;
        }

        form {
            display: flex;
            gap: 10px;
            margin-bottom: 30px;
        }

        input[type="url"] {
            flex: 1;
            padding: 12px;
            border: 1px solid #ccc;
            border-radius: 5px;
            font-size: 16px;
        }

        button {
            padding: 12px 18px;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            background: #222;
            color: white;
        }

        table {
            width: 100%;
            border-collapse: collapse;
        }

        th, td {
            padding: 12px;
            border-bottom: 1px solid #ddd;
            text-align: left;
        }

        th {
            background: #f1f1f1;
        }

        a {
            color: #0066cc;
            text-decoration: none;
        }

        .delete-btn {
            background: #c62828;
        }

        .url-column {
            max-width: 300px;
            word-break: break-all;
        }

    </style>

</head>

<body>

<div class="container">

    <h1>URL Shortener</h1>

    <form action="/shorten" method="POST">

        <input
            type="url"
            name="url"
            placeholder="Enter a long URL..."
            required
        >

        <button type="submit">
            Shorten URL
        </button>

    </form>


    {% if urls %}

    <table>

        <thead>

            <tr>
                <th>Original URL</th>
                <th>Short URL</th>
                <th>Clicks</th>
                <th>Action</th>
            </tr>

        </thead>

        <tbody>

        {% for url in urls %}

            <tr>

                <td class="url-column">
                    {{ url["original_url"] }}
                </td>

                <td>

                    <a href="/{{ url['short_code'] }}"
                       target="_blank">

                        {{ request.host_url }}{{ url["short_code"] }}

                    </a>

                </td>

                <td>
                    {{ url["clicks"] }}
                </td>

                <td>

                    <form
                        action="/delete/{{ url['short_code'] }}"
                        method="POST"
                        style="margin:0;">

                        <button
                            type="submit"
                            class="delete-btn">

                            Delete

                        </button>

                    </form>

                </td>

            </tr>

        {% endfor %}

        </tbody>

    </table>

    {% else %}

        <p style="text-align:center;">
            No shortened URLs yet.
        </p>

    {% endif %}

</div>

</body>

</html>